In [1]:
from pathlib import Path
import polars as pl

# Buat folder data/raw jika belum ada
DATA_DIR = Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / 'dataset.parquet'

# Buat dummy dataset sederhana menggunakan Polars murni
n_rows = 1000

df_dummy = pl.DataFrame({
    'ID': [f'ID_{i}' for i in range(n_rows)],
    'Delivery_person_Age': [25 if i % 2 == 0 else None for i in range(n_rows)],
    'Delivery_person_Ratings': [4.5 if i % 3 == 0 else 4.0 for i in range(n_rows)],
    'Weatherconditions': ['Sunny' if i % 2 == 0 else 'Stormy' for i in range(n_rows)],
    'Road_traffic_density': ['High' if i % 3 == 0 else 'Low' for i in range(n_rows)],
    'Festival': ['Yes' if i % 5 == 0 else 'No' for i in range(n_rows)],
    'City': ['Metropolitan' if i % 2 == 0 else 'Urban' for i in range(n_rows)],
    'Time_taken(min)': [20 + (i % 30) for i in range(n_rows)]
})

# Simpan ke format parquet
df_dummy.write_parquet(DATA_PATH)
print(f"File dataset berhasil dibuat di: {DATA_PATH}")

File dataset berhasil dibuat di: data/raw/dataset.parquet


In [2]:
from pathlib import Path
import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df = pl.read_parquet(DATA_PATH)
print(f"Ukuran awal dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom")

print("\n--- INFORMASI KOLOM & NULL VALUE ---")
null_counts = df.select([pl.col(c).null_count().alias(c) for c in df.columns])
print(null_counts)

df.head(5)

Ukuran awal dataset: 1,000 baris x 8 kolom

--- INFORMASI KOLOM & NULL VALUE ---
shape: (1, 8)
┌─────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────┬──────┬──────────────┐
│ ID  ┆ Delivery_per ┆ Delivery_per ┆ Weathercondi ┆ Road_traffic ┆ Festival ┆ City ┆ Time_taken(m │
│ --- ┆ son_Age      ┆ son_Ratings  ┆ tions        ┆ _density     ┆ ---      ┆ ---  ┆ in)          │
│ u32 ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ u32      ┆ u32  ┆ ---          │
│     ┆ u32          ┆ u32          ┆ u32          ┆ u32          ┆          ┆      ┆ u32          │
╞═════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════╪══════╪══════════════╡
│ 0   ┆ 500          ┆ 0            ┆ 0            ┆ 0            ┆ 0        ┆ 0    ┆ 0            │
└─────┴──────────────┴──────────────┴──────────────┴──────────────┴──────────┴──────┴──────────────┘


ID,Delivery_person_Age,Delivery_person_Ratings,Weatherconditions,Road_traffic_density,Festival,City,Time_taken(min)
str,i64,f64,str,str,str,str,i64
"""ID_0""",25,4.5,"""Sunny""","""High""","""Yes""","""Metropolitan""",20
"""ID_1""",null,4.0,"""Stormy""","""Low""","""No""","""Urban""",21
"""ID_2""",25,4.0,"""Sunny""","""Low""","""No""","""Metropolitan""",22
"""ID_3""",null,4.5,"""Stormy""","""High""","""No""","""Urban""",23
"""ID_4""",25,4.0,"""Sunny""","""Low""","""No""","""Metropolitan""",24


In [3]:
print("--- STATISTIK DESKRIPTIF DATA ---")
display(df.describe())

print("\n--- INFORMASI NILAI NULL PER KOLOM ---")
print(df.null_count())

--- STATISTIK DESKRIPTIF DATA ---


statistic,ID,Delivery_person_Age,Delivery_person_Ratings,Weatherconditions,Road_traffic_density,Festival,City,Time_taken(min)
str,str,f64,f64,str,str,str,str,f64
"""count""","""1000""",500.0,1000.0,"""1000""","""1000""","""1000""","""1000""",1000.0
"""null_count""","""0""",500.0,0.0,"""0""","""0""","""0""","""0""",0.0
"""mean""",null,25.0,4.167,null,null,null,null,34.4
"""std""",null,0.0,0.235938,null,null,null,null,8.67844
"""min""","""ID_0""",25.0,4.0,"""Stormy""","""High""","""No""","""Metropolitan""",20.0
"""25%""",null,25.0,4.0,null,null,null,null,27.0
"""50%""",null,25.0,4.0,null,null,null,null,34.0
"""75%""",null,25.0,4.5,null,null,null,null,42.0
"""max""","""ID_999""",25.0,4.5,"""Sunny""","""Low""","""Yes""","""Urban""",49.0



--- INFORMASI NILAI NULL PER KOLOM ---
shape: (1, 8)
┌─────┬──────────────┬──────────────┬──────────────┬──────────────┬──────────┬──────┬──────────────┐
│ ID  ┆ Delivery_per ┆ Delivery_per ┆ Weathercondi ┆ Road_traffic ┆ Festival ┆ City ┆ Time_taken(m │
│ --- ┆ son_Age      ┆ son_Ratings  ┆ tions        ┆ _density     ┆ ---      ┆ ---  ┆ in)          │
│ u32 ┆ ---          ┆ ---          ┆ ---          ┆ ---          ┆ u32      ┆ u32  ┆ ---          │
│     ┆ u32          ┆ u32          ┆ u32          ┆ u32          ┆          ┆      ┆ u32          │
╞═════╪══════════════╪══════════════╪══════════════╪══════════════╪══════════╪══════╪══════════════╡
│ 0   ┆ 500          ┆ 0            ┆ 0            ┆ 0            ┆ 0        ┆ 0    ┆ 0            │
└─────┴──────────────┴──────────────┴──────────────┴──────────────┴──────────┴──────┴──────────────┘


In [4]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""Delivery_person_Age""",500,50.0


In [5]:
# 3. Penanganan Missing Values
df_missing_handled = df.with_columns([
    # Mengisi kolom numerik dengan median
    pl.col('Delivery_person_Age').fill_null(pl.col('Delivery_person_Age').median()),
    # Kolom lainnya jika ada yang kosong bisa disesuaikan, misal rating atau teks
    pl.col('Delivery_person_Ratings').fill_null(pl.col('Delivery_person_Ratings').mean())
])

print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}")

Null sebelum: 500
Null sesudah: 0


In [6]:
# 4. Deteksi dan Penanganan Outlier menggunakan IQR & Winsorization
KOLOM_NUMERIK = 'Time_taken(min)'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Batas bawah: {batas_bawah}, Batas atas: {batas_atas}")

# Winsorization: membatasi nilai ekstrim ke batas valid tanpa menghapus baris
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

Q1=27.0, Q3=42.0, IQR=15.0
Batas bawah: 4.5, Batas atas: 64.5


In [7]:
# 5. Deduplikasi
KOLOM_KUNCI = 'ID'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}")
print(f"Duplikat: {total_baris - baris_unik:,}")

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

Total baris: 1,000
Baris unik berdasarkan ID: 1,000
Duplikat: 0


In [8]:
# 6. Validasi Data Dasar (Menggunakan Polars Native untuk menghindari error pyarrow)
validasi = pl.DataFrame({
    "total_baris": [len(df_dedup)],
    "pelanggaran_null": [df_dedup[KOLOM_NUMERIK].null_count()],
    "pelanggaran_batas": [df_dedup.filter((pl.col(KOLOM_NUMERIK) < batas_bawah) | (pl.col(KOLOM_NUMERIK) > batas_atas)).height]
})
validasi

total_baris,pelanggaran_null,pelanggaran_batas
i64,i64,i64
1000,0,0


In [9]:
# 7. Simpan Data Bersih
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")

Baris sebelum cleaning: 1,000
Baris sesudah cleaning: 1,000
Data bersih tersimpan di: /home/jovyan/work/data/processed/dataset_bersih.parquet


### Catatan Milestone 2

1. Dimensi kualitas data yang bermasalah :
   - Completeness (Kelengkapan): ditemukan missing values pada kolom Delivery_person_Age
   - Accuracy/Validity (Akurasi): terdapat nilai durasi pengiriman yang berpotensi menjadi outlier.

2. Strategi Penanganan:
   - Missing Values: melakukan imputasi menggunakan nilai median pada kolom Delivery_person_Age
   - Outlier: Menggunakan metode IQR dan teknik Winsorization (clip) untuk menangani nilai ekstrem tanpa menghapus baris
   - Deduplikasi: Memeriksa dan memastikan tidak ada data duplikat berdasarkan kunci unik ID

3. Aturan Validasi:
   - Memastikan bahwa seluruh baris data bersih tidak memiliki nilai null pada kolom utama serta berada dalam batas bawah dan batas atas yang valid.